# Feature Value Inspection
## Every single feature — its value, distribution, and health

Interactive notebook to inspect any feature in the V4 training dataset.
Run cells to see distributions, correlations, and anomalies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

# Load V4 dataset — change path if running on server
# Server: nba/features/datasets/xl_v2_matchup_training_POINTS_2023_2025.csv
df = pd.read_csv('nba/features/datasets/xl_v2_matchup_training_POINTS_2023_2025.csv')
print(f'Shape: {df.shape}')
print(f'Date range: {df["game_date"].min()} to {df["game_date"].max()}')

## ALL Features — Full Report Card

In [ ]:
# Build complete feature report
report = []
label = df['label'] if 'label' in df.columns else None

for col in sorted(df.columns):
    s = df[col]
    row = {
        'feature': col,
        'dtype': str(s.dtype),
        'nulls': s.isnull().sum(),
        'unique': s.nunique(),
    }
    if s.dtype in [np.float64, np.int64, float, int, bool]:
        row['mean'] = round(s.mean(), 4)
        row['std'] = round(s.std(), 4)
        row['min'] = s.min()
        row['max'] = s.max()
        row['pct_zero'] = round((s == 0).mean() * 100, 1)
        row['pct_nonzero'] = round((s != 0).mean() * 100, 1)
        if label is not None and s.std() > 0:
            row['corr_label'] = round(s.corr(label), 4)
        else:
            row['corr_label'] = 0
        # Health status
        if s.std() == 0:
            row['status'] = 'DEAD (zero-variance)'
        elif (s == 0).mean() > 0.9:
            row['status'] = 'SPARSE (>90% zero)'
        elif (s == 0).mean() > 0.5:
            row['status'] = 'WEAK (>50% zero)'
        else:
            row['status'] = 'OK'
    else:
        row['mean'] = '-'
        row['std'] = '-'
        row['min'] = s.iloc[0] if len(s) > 0 else '-'
        row['max'] = '-'
        row['pct_zero'] = '-'
        row['pct_nonzero'] = '-'
        row['corr_label'] = '-'
        row['status'] = 'META'
    report.append(row)

rdf = pd.DataFrame(report)
print(f'Total features: {len(rdf)}')
print(f'DEAD: {(rdf["status"] == "DEAD (zero-variance)").sum()}')
print(f'SPARSE: {(rdf["status"] == "SPARSE (>90% zero)").sum()}')
print(f'WEAK: {(rdf["status"] == "WEAK (>50% zero)").sum()}')
print(f'OK: {(rdf["status"] == "OK").sum()}')
print(f'META: {(rdf["status"] == "META").sum()}')
rdf

## Dead Features (zero-variance)

In [ ]:
dead = rdf[rdf['status'] == 'DEAD (zero-variance)'][['feature','mean','status']]
print(f'{len(dead)} dead features:')
dead

## Top Correlated Features with Label

In [ ]:
corr_df = rdf[rdf['corr_label'] != '-'].copy()
corr_df['abs_corr'] = corr_df['corr_label'].astype(float).abs()
top_corr = corr_df.sort_values('abs_corr', ascending=False).head(40)
print('Top 40 features by correlation with label:')
top_corr[['feature','corr_label','pct_nonzero','mean','std','status']]

## Suspicious Features (anomalies)

In [ ]:
# num_books_offering — should be 1-10, check max
if 'num_books_offering' in df.columns:
    nbo = df['num_books_offering']
    print(f'num_books_offering: min={nbo.min()}, max={nbo.max()}, mean={nbo.mean():.1f}')
    print(f'  Values > 10: {(nbo > 10).sum()} ({(nbo > 10).mean()*100:.1f}%)')
    print(f'  Values > 50: {(nbo > 50).sum()}')
    plt.figure(figsize=(10,4))
    plt.hist(nbo.clip(upper=50), bins=50)
    plt.title('num_books_offering distribution (clipped at 50)')
    plt.show()

# Duplicate features
numeric = df.select_dtypes(include='number')
corr_matrix = numeric.corr()
dupes = []
for i, c1 in enumerate(corr_matrix.columns):
    for c2 in corr_matrix.columns[i+1:]:
        if abs(corr_matrix.loc[c1, c2]) > 0.999:
            dupes.append((c1, c2, round(corr_matrix.loc[c1, c2], 4)))
print(f'\nPerfectly correlated pairs (r > 0.999): {len(dupes)}')
for a, b, r in dupes:
    print(f'  {a} <-> {b}: r={r}')

## H2H Features Deep Dive

In [ ]:
h2h = rdf[rdf['feature'].str.startswith('h2h_')][['feature','mean','std','pct_zero','pct_nonzero','corr_label','status']]
print(f'{len(h2h)} H2H features:')
h2h

## V4 New Features (not in V3)

In [ ]:
import pickle
try:
    v3_feats = set(pickle.load(open('nba/models/saved_xl/points_v3_features.pkl', 'rb')))
    v4_only = rdf[~rdf['feature'].isin(v3_feats) & (rdf['status'] != 'META')]
    print(f'{len(v4_only)} V4-only features:')
    v4_only[['feature','mean','std','pct_nonzero','corr_label','status']].sort_values('corr_label', key=abs, ascending=False)
except:
    print('Could not load V3 features — run this on a machine with the model files')

## Opponent Team Check

In [ ]:
if 'opponent_team' in df.columns:
    opp = df['opponent_team']
    print(f'Unique values: {opp.nunique()}')
    print(f'Nulls: {opp.isnull().sum()}')
    print(f'\nAll values:')
    print(opp.value_counts().to_string())

## Interactive Feature Inspector

In [ ]:
def inspect(feature_name):
    """Deep inspect a single feature"""
    if feature_name not in df.columns:
        print(f'{feature_name} not found')
        return
    s = df[feature_name]
    print(f'Feature: {feature_name}')
    print(f'  dtype: {s.dtype}')
    print(f'  nulls: {s.isnull().sum()}')
    print(f'  unique: {s.nunique()}')
    if s.dtype in [np.float64, np.int64, float, int]:
        print(f'  mean: {s.mean():.4f}')
        print(f'  std: {s.std():.4f}')
        print(f'  min: {s.min()}, max: {s.max()}')
        print(f'  zero: {(s==0).sum()} ({(s==0).mean()*100:.1f}%)')
        if label is not None:
            print(f'  corr with label: {s.corr(label):.4f}')
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].hist(s.dropna(), bins=50, alpha=0.7)
        axes[0].set_title(f'{feature_name} distribution')
        if label is not None:
            axes[1].boxplot([s[label==0].dropna(), s[label==1].dropna()], labels=['UNDER','OVER'])
            axes[1].set_title(f'{feature_name} by outcome')
        plt.tight_layout()
        plt.show()
    else:
        print(f'  values: {s.value_counts().head(10).to_dict()}')

# Usage: inspect('h2h_avg_points')
inspect('num_books_offering')

In [ ]:
# Inspect key features
for f in ['is_home', 'h2h_avg_points', 'h2h_std_points', 'bp_hit_rate_season', 
          'bp_analytics_hit_rate_season', 'dvp_stat_allowed', 'player_plus_minus_L5',
          'season_pct', 'player_games_with_team']:
    inspect(f)